# 01. Window max

**Tier:** Deep. **Why you'd use this:** Every window of length `k` needs its maximum. Rescanning each window costs `k` comparisons; a deque of candidate indexes answers in amortized constant time per step.

**After this lesson you can:** keep a decreasing deque of indexes inside the window; read the max of every window of size `k` in one pass.
About 35 minutes. **Uses:** [Deque](../../course/08-collections-module/03-deque-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. Why store indexes in the deque instead of values?
2. When does an index leave the front of the deque?

<details><summary>Answers</summary>

Indexes tell you when a candidate has slid out of the window. The front leaves when `dq[0] <= i - k`. Skim [Deque](../../course/08-collections-module/03-deque-lesson.ipynb) if needed.
</details>


## Recognize it

You need the maximum (or minimum) of every contiguous block of exactly `k` values. The prompt talks about "sliding window maximum", "max of each subarray of length k", or streaming extrema over a fixed span.

Signals: fixed window length, an extreme (max or min) per window, and online answers as the window slides. A look-alike that is not this pattern: the maximum *sum* of a k-window — that is a running sum, not a monotonic deque. A variable-length window with a sum constraint is sliding window of a different kind.


## Why it works this way

For window max, the answer at the right edge `i` is the largest value among `nums[i - k + 1 : i + 1]`. A nested scan walks those `k` values for every `i`. Across roughly `n` windows the comparisons grow like `n · k`. When `k` is thousands and `n` is hundreds of thousands, that cost shows up immediately.

A monotonic decreasing deque stores indexes whose values go from large to small, front to back. The front is always the index of the current maximum. Three updates keep the invariant when `i` advances:

1. Drop the front while its index is outside the window (`dq[0] <= i - k`).
2. Drop from the back while `nums[dq[-1]]` is less than or equal to the new value. Those indexes can never be the max while the newcomer is still inside the window.
3. Append `i`. If `i >= k - 1`, the front value is this window's answer.

Each index is pushed once and popped at most once, so total deque work across the whole array is proportional to `n`, not to `n · k`. Extra memory is at most `k` indexes. See [Deque](../../course/08-collections-module/03-deque-lesson.ipynb) for why `popleft` and `pop` stay cheap.

Storing values alone is not enough. When equals appear, or when the max slides out, you need the index to know expiry. The comparison `<=` (not `<`) drops older equals so the front is the rightmost max; either strictness can be valid if the prompt does not care which equal wins, but you must pick one rule and keep it.

A heap of `(value, index)` pairs also tracks the max, but lazy deletions cost logarithmic factors and more code. The deque wins when you need every window answer in linear time. Prefer a plain rescan only for tiny `k` where clarity matters more than the linear bound.


## When to use it

Use a monotonic deque when every fixed window needs its max or min and `n · k` is too large. Flip the back-pop comparison for minima. Use a running sum for window sums or averages. Use two pointers or a variable window when the length itself is part of the answer. Use a heap when you need the top-k values inside a changing set that is not a contiguous window.


## Brute force, and why it's slow

For each right edge, scan the whole window for the max. Count the comparisons.


In [1]:
def window_max_slow(nums, k):
    comparisons = 0
    out = []
    for right in range(k - 1, len(nums)):
        best = nums[right - k + 1]
        for index in range(right - k + 2, right + 1):
            comparisons += 1
            if nums[index] > best:
                best = nums[index]
        out.append(best)
    return out, comparisons


for size, k in ((50, 10), (200, 20), (800, 40)):
    data = list(range(size))
    _out, comparisons = window_max_slow(data, k)
    print(size, k, comparisons)

50 10 369
200 20 3439
800 40 29679


## Key insight

Keep indexes in decreasing value order. The front is the max of the current window. Drop expired fronts and dominated backs before you append the new index — each index enters and leaves once.


## Worked example

Return the maximum of every contiguous window of length `k`.


In [2]:
from collections import deque


def window_max(nums: list[int], k: int) -> list[int]:
    # 1. Decreasing indexes: front is the max of the live window.
    out = []
    dq = deque()
    for i, value in enumerate(nums):
        # 2. Drop indexes that slid out on the left.
        while dq and dq[0] <= i - k:
            dq.popleft()
        # 3. Drop backs that can never beat this value.
        while dq and nums[dq[-1]] <= value:
            dq.pop()
        dq.append(i)
        # 4. Once the first full window exists, record the front.
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))
print(window_max([1, -1], 1))

[3, 3, 5, 5, 6, 7]
[1, -1]


## Trace

`nums = [1, 3, -1, -3, 5]`, `k = 3`:

| i | value | deque indexes (values) | out |
|---|---|---|---|
| 0 | 1 | `[0]` (1) | — |
| 1 | 3 | `[1]` (3) | — |
| 2 | -1 | `[1, 2]` (3, -1) | **3** |
| 3 | -3 | `[1, 2, 3]` (3, -1, -3) | **3** |
| 4 | 5 | `[4]` (5) | **5** |


## Cost

One pass: each index is pushed and popped at most once, so time grows with `n`. Extra memory is at most `k` indexes. That beats the nested window scan because you never recompare a dominated value while a better candidate remains inside the window.


## Variants

Practice flips `<=` to `>=` for window minima. Related challenges (not solved here): [Daily Temperatures](../../challenges/problems/23-daily-temperatures.ipynb) uses a monotonic stack of indexes, not a sliding window deque.


## Common mistake

This bug is silent. The code drops dominated backs and appends the new index, but never expires the front when it slides out of the window. On `[2, 1, 3, 0]` with `k = 2` the last window is `[3, 0]` so the max is `3`, yet a stale front from an earlier larger value can survive if you skip expiry — here the broken run prints a wrong list because index `0` (value `2`) is never removed when the window has moved on.


In [3]:
def window_max_wrong(nums, k):
    out = []
    dq = deque()
    for i, value in enumerate(nums):
        # missing: while dq and dq[0] <= i - k: dq.popleft()
        while dq and nums[dq[-1]] <= value:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


print(window_max_wrong([5, 1, 3, 0], 2))

[5, 5, 5]


The fix drops the front while `dq[0] <= i - k` before reading the answer. The worked example does that; for `[5, 1, 3, 0]` with `k = 2` it prints `[5, 3, 3]`.


In [4]:
print(window_max([5, 1, 3, 0], 2))

[5, 3, 3]


## Python idioms

`collections.deque` gives `O(1)` `popleft` and `pop`. A list's `pop(0)` would copy the rest of the list and destroy the linear bound. `enumerate` pairs each value with the index the deque stores.


In [5]:
def window_max_idiom(nums: list[int], k: int) -> list[int]:
    dq: deque[int] = deque()
    out = []
    for i, value in enumerate(nums):
        while dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] <= value:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out


print(window_max_idiom([1, 3, -1, -3, 5], 3))

[3, 3, 5]


## Recap

- A decreasing deque of indexes keeps the window max at the front.
- Drop expired fronts and dominated backs before you append.
- Each index enters and leaves once, so the pass is linear in `n`.

Next: [practice](01-window-max-practice.ipynb).
